In [1]:
# ============================================================
# CodeAlpha — Task 3: Data Visualization
# Dataset: Titanic
# All charts saved to images/ — no inline display
# ============================================================

# ---------- 1. Imports ----------
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.max_open_warning'] = 0
plt.rcParams['font.family'] = 'DejaVu Sans'

IMG_DIR = "images"
DATA_DIR = "data"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ---------- 2. Load & Prepare Data ----------
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Same cleaning as Task 2 (keeps your portfolio consistent)
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['HasCabin'] = df['Cabin'].notna().astype(int)

# Friendly labels
df['Sex'] = df['Sex'].str.capitalize()
df['Pclass'] = df['Pclass'].map({1: '1st Class', 2: '2nd Class', 3: '3rd Class'})
df['SurvivedLabel'] = df['Survived'].map({1: 'Survived', 0: 'Died'})
df['Embarked'] = df['Embarked'].map({'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'})
df['AgeGroup'] = pd.cut(df['Age'],
                        bins=[0, 12, 18, 30, 50, 80],
                        labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])

# Color palette used across all charts
PALETTE = {'Survived': '#2ecc71', 'Died': '#e74c3c'}
CLASS_COLORS = ['#f1c40f', '#3498db', '#9b59b6']

# ============================================================
# CHART 1 — Overall Survival Rate (Donut)
# ============================================================
surv_counts = df['SurvivedLabel'].value_counts()
fig, ax = plt.subplots(figsize=(8, 6))
colors = [PALETTE['Survived'], PALETTE['Died']]
wedges, texts, autotexts = ax.pie(
    surv_counts,
    labels=surv_counts.index,
    autopct=lambda p: f'{p:.1f}%\n({int(p*len(df)/100)} people)',
    colors=colors,
    startangle=90,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
    textprops={'fontsize': 12, 'weight': 'bold'}
)
ax.set_title("Overall Survival Rate — Titanic", fontsize=15, weight='bold', pad=20)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/01_survival_donut.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 2 — Survival Rate by Sex & Class (Grouped Bar)
# ============================================================
pivot = df.groupby(['Pclass', 'Sex'])['Survived'].mean().reset_index()
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=pivot, x='Pclass', y='Survived', hue='Sex',
            palette={'Male': '#3498db', 'Female': '#e91e63'}, ax=ax)
ax.set_title("Survival Rate by Passenger Class & Sex",
             fontsize=15, weight='bold')
ax.set_xlabel("Passenger Class", fontsize=12)
ax.set_ylabel("Survival Rate", fontsize=12)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Annotate values
for container in ax.containers:
    ax.bar_label(container, fmt='%.0%%', padding=3, fontsize=10)
ax.legend(title='Sex', loc='upper left')
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/02_survival_by_class_sex.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 3 — Age Distribution by Survival (Overlaid Histogram)
# ============================================================
fig, ax = plt.subplots(figsize=(11, 6))
sns.histplot(data=df, x='Age', hue='SurvivedLabel',
             bins=30, kde=True, palette=PALETTE,
             alpha=0.6, ax=ax)
ax.set_title("Age Distribution by Survival", fontsize=15, weight='bold')
ax.set_xlabel("Age", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/03_age_distribution.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 4 — Survival by Age Group (Bar + Count)
# ============================================================
age_pivot = df.groupby('AgeGroup', observed=True)['Survived'].agg(['mean', 'count']).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 6))
bars = ax1.bar(age_pivot['AgeGroup'], age_pivot['mean'],
               color='#2ecc71', alpha=0.85, edgecolor='black')
ax1.set_ylabel("Survival Rate", fontsize=12, color='#2ecc71')
ax1.set_xlabel("Age Group", fontsize=12)
ax1.set_ylim(0, 1)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax1.tick_params(axis='y', labelcolor='#2ecc71')

# Annotate % on bars
for bar, val in zip(bars, age_pivot['mean']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.0%}', ha='center', fontsize=11, weight='bold')

# Count line on secondary axis
ax2 = ax1.twinx()
ax2.plot(age_pivot['AgeGroup'], age_pivot['count'],
         color='#34495e', marker='o', linewidth=2.5, markersize=9)
ax2.set_ylabel("Number of Passengers", fontsize=12, color='#34495e')
ax2.tick_params(axis='y', labelcolor='#34495e')

ax1.set_title("Survival Rate & Passenger Count by Age Group",
              fontsize=15, weight='bold')
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/04_survival_by_age_group.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 5 — Fare Distribution (Box Plot by Class & Survival)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Pclass', y='Fare', hue='SurvivedLabel',
            palette=PALETTE, ax=ax)
ax.set_title("Fare Distribution by Class & Survival",
             fontsize=15, weight='bold')
ax.set_xlabel("Passenger Class", fontsize=12)
ax.set_ylabel("Fare (£)", fontsize=12)
ax.set_yscale('log')
ax.legend(title='Outcome')
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/05_fare_by_class.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 6 — Family Size vs Survival (Feature Engineering)
# ============================================================
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
fam_pivot = df.groupby('FamilySize')['Survived'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=fam_pivot, x='FamilySize', y='mean',
            hue='FamilySize', palette='viridis', legend=False, ax=ax)
ax.set_title("Survival Rate by Family Size",
             fontsize=15, weight='bold')
ax.set_xlabel("Family Size (including self)", fontsize=12)
ax.set_ylabel("Survival Rate", fontsize=12)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Annotate survival % and count
for i, row in fam_pivot.iterrows():
    ax.text(i, row['mean'] + 0.02,
            f"{row['mean']:.0%}\n(n={int(row['count'])})",
            ha='center', fontsize=9, weight='bold')

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/06_family_size.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 7 — Correlation Heatmap (Focused Features)
# ============================================================
feature_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch',
                'Fare', 'HasCabin', 'FamilySize']
corr = df[feature_cols].copy()
corr['Pclass'] = df['Pclass'].map({'1st Class': 1, '2nd Class': 2, '3rd Class': 3})
corr_mat = corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title("Correlation Heatmap — Key Features",
             fontsize=15, weight='bold', pad=15)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/07_correlation.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 8 — Embarkation Port vs Survival (Stacked Bar)
# ============================================================
emb = pd.crosstab(df['Embarked'], df['SurvivedLabel'], normalize='index') * 100
emb = emb[['Survived', 'Died']]

fig, ax = plt.subplots(figsize=(10, 6))
emb.plot(kind='barh', stacked=True, color=[PALETTE['Survived'], PALETTE['Died']],
         ax=ax, edgecolor='white', linewidth=1.5)
ax.set_title("Survival Rate by Embarkation Port",
             fontsize=15, weight='bold')
ax.set_xlabel("Percentage (%)", fontsize=12)
ax.set_ylabel("Port", fontsize=12)
ax.legend(title='Outcome', loc='lower right')
ax.set_xlim(0, 100)

# Annotate
for i, (idx, row) in enumerate(emb.iterrows()):
    ax.text(row['Survived'] / 2, i, f"{row['Survived']:.0f}%",
            ha='center', va='center', fontsize=11,
            weight='bold', color='white')
    ax.text(row['Survived'] + row['Died'] / 2, i, f"{row['Died']:.0f}%",
            ha='center', va='center', fontsize=11,
            weight='bold', color='white')

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/08_embarked_survival.png", bbox_inches='tight')
plt.close()

# ============================================================
# CHART 9 — Dashboard: 4-panel overview
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Titanic — Survival Dashboard", fontsize=18, weight='bold', y=1.02)

# Panel A — Survival by Sex
sex_pivot = df.groupby('Sex')['Survived'].mean()
axes[0, 0].bar(sex_pivot.index, sex_pivot.values,
               color=['#3498db', '#e91e63'], edgecolor='black')
axes[0, 0].set_title("Survival Rate by Sex", weight='bold')
axes[0, 0].set_ylim(0, 1)
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
for i, v in enumerate(sex_pivot.values):
    axes[0, 0].text(i, v + 0.03, f'{v:.0%}', ha='center', weight='bold')

# Panel B — Survival by Class
cls_pivot = df.groupby('Pclass')['Survived'].mean().reindex(['1st Class', '2nd Class', '3rd Class'])
axes[0, 1].bar(cls_pivot.index, cls_pivot.values,
               color=CLASS_COLORS, edgecolor='black')
axes[0, 1].set_title("Survival Rate by Class", weight='bold')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
for i, v in enumerate(cls_pivot.values):
    axes[0, 1].text(i, v + 0.03, f'{v:.0%}', ha='center', weight='bold')

# Panel C — Age distribution
axes[1, 0].hist([df[df['Survived'] == 0]['Age'].dropna(),
                 df[df['Survived'] == 1]['Age'].dropna()],
                bins=25, color=['#e74c3c', '#2ecc71'],
                label=['Died', 'Survived'], alpha=0.75)
axes[1, 0].set_title("Age Distribution by Survival", weight='bold')
axes[1, 0].legend()
axes[1, 0].set_xlabel("Age")

# Panel D — Fare distribution
axes[1, 1].hist([df[df['Survived'] == 0]['Fare'].dropna(),
                 df[df['Survived'] == 1]['Fare'].dropna()],
                bins=30, color=['#e74c3c', '#2ecc71'],
                label=['Died', 'Survived'], alpha=0.75)
axes[1, 1].set_title("Fare Distribution by Survival", weight='bold')
axes[1, 1].set_xscale('log')
axes[1, 1].legend()
axes[1, 1].set_xlabel("Fare (log scale)")

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/09_dashboard.png", bbox_inches='tight')
plt.close()

# ============================================================
# SUMMARY
# ============================================================
print("=" * 60)
print("DATA VISUALIZATION — COMPLETE")
print("=" * 60)
print(f"\n✅ 9 charts saved to /{IMG_DIR}/:")
for f in sorted(os.listdir(IMG_DIR)):
    if f.endswith(".png"):
        print(f"   • {f}")

print("\n📌 Key Story Insights:")
print("   1. Women survived at ~74%, men at ~19% — sex is the #1 factor.")
print("   2. 1st class = 63% survival, 3rd class = 24% — class matters.")
print("   3. Children had the highest survival rate by age group.")
print("   4. Small families (2–4) survived better than solo travelers.")
print("   5. Survivors paid significantly higher fares.")
print("   6. Cherbourg passengers had the best survival odds.")

DATA VISUALIZATION — COMPLETE

✅ 9 charts saved to /images/:
   • 01_survival_donut.png
   • 02_survival_by_class_sex.png
   • 03_age_distribution.png
   • 04_survival_by_age_group.png
   • 05_fare_by_class.png
   • 06_family_size.png
   • 07_correlation.png
   • 08_embarked_survival.png
   • 09_dashboard.png

📌 Key Story Insights:
   1. Women survived at ~74%, men at ~19% — sex is the #1 factor.
   2. 1st class = 63% survival, 3rd class = 24% — class matters.
   3. Children had the highest survival rate by age group.
   4. Small families (2–4) survived better than solo travelers.
   5. Survivors paid significantly higher fares.
   6. Cherbourg passengers had the best survival odds.
